python scripts/reformat_sabin.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_test.tsv

In [37]:
import pandas as pd

In [38]:
df_combined = pd.read_csv('../combined_test_einstein.tsv', sep='\t')

In [39]:
df_sabin_covid = pd.read_excel(
    './../data/SABIN/20230711_SABIN_Covid_2023ateSE27.xlsx'
)

df_dict = pd.read_excel(
    './../data/SABIN/20230711_SABIN_paineis_2023ateSE27.xlsx', 
    index_col=None, header=0, 
    sheet_name=None, 
    dtype='str'
)

df_paineis = pd.DataFrame()
for sheet in df_dict.keys():
    if sheet.startswith('EXAME'):
        # einstein
        continue
    
    df_paineis = df_paineis.append(df_dict[sheet].assign(ExcelSheet=sheet))

In [40]:
df_full_raw = df_sabin_covid.append(df_paineis)

## Duplicatas

In [41]:
df_duplicates = (
    df_combined
    # .query("test_kit == 'unknown'")
    .assign( one=1 )
    .groupby(['test_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

10587 2 1


,num_duplicates
test_id,
2022020007232,2
2023199007171,2
2022027000933,2
2022020008542,2
2022127002379,2
2022274007730,2
2022013011347,2
2022017006199,2
2022025007903,2


In [42]:
df_os_multiple_tests = (
    df_combined
    .groupby('test_id')
    .agg(kits=('test_kit',lambda x: ','.join(x.tolist())))
    .query("kits.str.contains(',')", engine='python') 
)


In [43]:
df_os_multiple_tests.head(10)

,kits
test_id,
2022001001776,"test_1,test_2"
2022001002045,"test_1,test_2"
2022001002070,"test_1,test_2"
2022001002079,"test_1,test_2"
2022001002390,"test_1,test_2"
2022001002836,"test_1,test_2"
2022001002930,"test_1,test_2"
2022001002950,"test_1,test_2"
2022001003041,"test_1,test_2"


In [44]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

## Uniqueness

In [45]:
if 'unknown' in df_combined['test_kit'].unique():
    print('WARNING! unknown test kits found')

df_combined['test_kit'].unique()

array(['test_1', 'test_3', 'test_2', 'test_4'], dtype=object)

In [46]:
df_combined['state'].unique()

array(['SÃO PAULO', 'RIO DE JANEIRO', 'GOIÁS'], dtype=object)

In [47]:
df_combined['location'].unique()

array(['GUARULHOS', 'SÃO PAULO', 'RIO DE JANEIRO', 'ALPHAVILLE',
       'SOROCABA', 'GOIÂNIA'], dtype=object)

In [48]:
df_combined['date_testing'].max(), df_combined['date_testing'].min()

('2023-07-24', '2022-01-01')

In [49]:
df_combined['age'].unique(), df_combined['age'].min(), df_combined['age'].max()

(array([ 61,  53,  58,  49,  17,  56,  12,  16,  45,  26,  54,  14,  23,
         36,  48,  51,  50,  82,  59,  15,  55,  22,  35,  27,  60,  46,
          1,   3,  24,  37,  28,  42,  20,  41,  10,   8,  21,  40,   4,
         39,  34,  33,   6,  30,  87,  43,   7,  32,  38,  66,  67,  74,
         25,  79,  65,  19,  31,  29,  11,   9,  62,  44,   2,  47,  57,
         18,  52,  84,  64,   5,  69,  70,  75,  63,  71,  72,  77,  13,
         86,  73,  68,  93,  83,  80,  85,  76,  89,  94,  95,  81,  97,
         96,  88,  78,  91, 101, 100,  92,  99,  98,  90, 103, 102, 110,
        104, 107, 108, 105,   0, 106, 112]),
 0,
 112)

In [50]:
df_combined['sex'].unique()

array(['M', 'F', 'I'], dtype=object)

In [51]:
for column in df_combined.columns:
    if column.endswith('_result'):
        print(column, df_combined[column].unique())

FLUA_test_result ['NT' 'Neg' 'Pos']
FLUB_test_result ['NT' 'Neg' 'Pos']
VSR_test_result ['NT' 'Neg' 'Pos']
SC2_test_result ['Neg' 'Pos' 'NT']
META_test_result ['NT']
RINO_test_result ['NT']
PARA_test_result ['NT']
ADENO_test_result ['NT']
BOCA_test_result ['NT']
COVS_test_result ['NT']
ENTERO_test_result ['NT']
BAC_test_result ['NT']


In [52]:
for col in df_combined.columns:
    if col.endswith('_result'):
        print(col)

FLUA_test_result
FLUB_test_result
VSR_test_result
SC2_test_result
META_test_result
RINO_test_result
PARA_test_result
ADENO_test_result
BOCA_test_result
COVS_test_result
ENTERO_test_result
BAC_test_result


## Inconsistencies

In [53]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

Verificando se um mesmo test_id + test_kit apresentam mais de um resultado para um mesmo patógeno

In [54]:
agg_test_rules = {
    col+'_kit': (col, lambda x: ','.join(x.tolist()))
    for col in df_combined.columns
    if col.endswith('_result')
}

df_os_non_contraditory_test_results = (
    df_combined
    .groupby(['test_id', 'test_kit'])
    .agg(**agg_test_rules)
)

In [55]:
for col in df_os_non_contraditory_test_results.columns:
    if col.endswith('_kit'):
        print(col, df_os_non_contraditory_test_results[col].unique())
        # Devem ser apenas ['NT' 'Neg' 'Pos']

FLUA_test_result_kit ['NT' 'Neg' 'Pos']
FLUB_test_result_kit ['NT' 'Neg' 'Pos']
VSR_test_result_kit ['NT' 'Neg' 'Pos']
SC2_test_result_kit ['Neg' 'Pos' 'NT']
META_test_result_kit ['NT']
RINO_test_result_kit ['NT']
PARA_test_result_kit ['NT']
ADENO_test_result_kit ['NT']
BOCA_test_result_kit ['NT']
COVS_test_result_kit ['NT']
ENTERO_test_result_kit ['NT']
BAC_test_result_kit ['NT']


Verificando se todos os Ids originais etão presentes no arquivo final

In [56]:
set_test_ids_final_df = set(df_combined['test_id'])
set_test_ids_raw_data = set(df_full_raw['OS'])

In [57]:
# Testar se todos os test_ids da planilha original estão na planilha final
# deve retornar um set vazio
set_test_ids_raw_data.difference(set_test_ids_final_df)

{'830-66532-15798',
 '936-66593-16207',
 '223-66548-3453',
 '833-66631-6546',
 '1269-66494-7745',
 '054-66502-11870',
 '834-66478-16826',
 '960-66521-18899',
 '575-66557-18405',
 '589-66560-12090',
 '743-66597-15489',
 '562-66624-14484',
 '960-66528-12934',
 '237-66618-8854',
 '959-66647-12030',
 '586-66494-19404',
 '830-66659-17108',
 '772-66555-17600',
 '221-66643-2095',
 '1269-66625-66',
 '223-66475-569',
 '566-66656-6493',
 '570-66505-11529',
 '142-66604-11313',
 '138-66533-14150',
 '770-66562-20301',
 '589-66542-62',
 '466-66484-13542',
 '960-66613-15141',
 '960-66598-21848',
 '959-66640-4488',
 '221-66527-13358',
 '589-66530-15877',
 '025-66513-5855',
 '942-66648-8496',
 '584-66563-19902',
 '977-66659-18532',
 '223-66581-11409',
 '350-66513-10423',
 '844-66618-21328',
 '1858-66607-14839',
 '1041-66540-14831',
 '939-66566-2646',
 '844-66537-15537',
 '025-66640-15004',
 '772-66630-18855',
 '589-66610-22156',
 '1269-66510-2286',
 '901-66614-9370',
 '206-66591-17765',
 '936-66628-149

In [58]:
# Verificar se há test_ids na planilha final que não estão na planilha original
# deve retornar um set vazio
set_test_ids_final_df.difference(set_test_ids_raw_data)

{2022032015360,
 2022032015362,
 2022032015363,
 2022032015364,
 2022032015365,
 2022075006981,
 2022183010308,
 2022032015370,
 2022334005265,
 2022032015378,
 2022334005266,
 2022075007000,
 2022032015386,
 2022075007003,
 2023068008485,
 2022032015398,
 2023068008486,
 2022032015402,
 2022032015403,
 2022032015407,
 2022032015409,
 2022183010355,
 2022183010356,
 2022334005302,
 2022334005304,
 2023068008504,
 2023068008506,
 2022032015420,
 2022075007041,
 2022032015427,
 2023068008516,
 2022334005318,
 2022032015433,
 2023068008524,
 2022032015451,
 2022032015452,
 2022183010400,
 2022226002046,
 2023068008574,
 2022226002053,
 2022183010448,
 2022334005395,
 2022334005396,
 2022334005398,
 2022226002075,
 2022183010470,
 2022226002092,
 2022183010487,
 2022183010492,
 2022075007166,
 2022183010505,
 2022183010508,
 2023068008653,
 2022183010514,
 2022183010521,
 2022075007200,
 2022075007212,
 2022334005484,
 2022226002168,
 2022334005499,
 2022075007244,
 2023068008723,
 2023068

Verificando se todos os patógenos de um test_kit estão sendo testados para todos os test_id

In [59]:
PATHOGENS_TESTS = {
    'panel_21':{
        # PAINCOVI
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'SC2_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'COVS_test_result',
        'BAC_test_result',
    },

    'panel_24':{
        # RESPIRA
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'BOCA_test_result',
        'COVS_test_result',
        'ENTERO_test_result',
        'BAC_test_result',
    },

    'panel_4':{
        # PCRESPSL & PCRVRESP
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'SC2_test_result',
    },

    'covid_antigen':{
        'SC2_test_result',
    },

    'covid_pcr':{
        'SC2_test_result',
    },
}

In [60]:
# Mapeando Pos e Neg para Tes
df_combined_test_pathogen_non_nt_on_kit = (
    df_combined
    .replace(('Pos', 'Neg'),  'Tes')
)

In [61]:
for pathogen, test_columns in PATHOGENS_TESTS.items():
    print(pathogen)
    print(
        df_combined_test_pathogen_non_nt_on_kit
        .query("test_kit == @pathogen")
        [list(test_columns)]
        .drop_duplicates().T
        # Deve resultar em apenas uma linha completa por 'Tes'
    )

    print('\n\n')

panel_21
Empty DataFrame
Columns: []
Index: [FLUB_test_result, FLUA_test_result, VSR_test_result, BAC_test_result, SC2_test_result, PARA_test_result, RINO_test_result, META_test_result, COVS_test_result, ADENO_test_result]



panel_24
Empty DataFrame
Columns: []
Index: [FLUB_test_result, FLUA_test_result, VSR_test_result, BAC_test_result, PARA_test_result, BOCA_test_result, RINO_test_result, META_test_result, COVS_test_result, ADENO_test_result, ENTERO_test_result]



panel_4
Empty DataFrame
Columns: []
Index: [FLUB_test_result, FLUA_test_result, SC2_test_result, VSR_test_result]



covid_antigen
Empty DataFrame
Columns: []
Index: [SC2_test_result]



covid_pcr
Empty DataFrame
Columns: []
Index: [SC2_test_result]





Verificando resultados de alguns testes

In [62]:
import numpy as np

In [63]:
np.random.seed(214)

ids = np.random.choice(df_combined['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [64]:
current_id=50

In [65]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_full_raw.query("OS == @id") [['OS', 'Parametro', 'Resultado']].T )

                            50298
test_id             2022011003051
test_kit                   test_2
FLUA_test_result              Neg
FLUB_test_result              Neg
VSR_test_result                NT
SC2_test_result                NT
META_test_result               NT
RINO_test_result               NT
PARA_test_result               NT
ADENO_test_result              NT
BOCA_test_result               NT
COVS_test_result               NT
ENTERO_test_result             NT
BAC_test_result                NT
Empty DataFrame
Columns: []
Index: [OS, Parametro, Resultado]


Verificando resultados de alguns testes - Fltrando por test_kit

In [66]:
np.random.seed(214)

ids = np.random.choice(df_combined.query("test_kit not in ('covid_pcr', 'covid_antigen')")['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [67]:
current_id=0

In [68]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_full_raw.query("OS == @id") [['OS', 'Parametro', 'Resultado']].T )

                            18113
test_id             2022005002687
test_kit                   test_2
FLUA_test_result              Neg
FLUB_test_result              Neg
VSR_test_result                NT
SC2_test_result                NT
META_test_result               NT
RINO_test_result               NT
PARA_test_result               NT
ADENO_test_result              NT
BOCA_test_result               NT
COVS_test_result               NT
ENTERO_test_result             NT
BAC_test_result                NT
Empty DataFrame
Columns: []
Index: [OS, Parametro, Resultado]
